# Simulating a full refresh, to see what the storage does

When pipeline writes get slower at certain times of day, the question is whether
the **storage** is the limit or the **workload** grew. Those need different
answers and a maximum settles neither — it says what the disk could do, not that
you were at it.

This runs a synthetic load shaped like a refresh, and reports what the device did
under it:

| Phase | Stands in for |
|---|---|
| `bulk` | large sequential writes — Spark writing parquet / Delta files |
| `commit` | small writes, each fsynced — Postgres WAL, the Delta log |
| `read` | sequential read back — a query scanning what was just written |
| `mixed` | both at once — what a refresh actually does, and the only phase where contention shows |

The work is done by **`iobench.sh`**, which is POSIX `sh` and coreutils only: no
Python, no packages, nothing to install. That is deliberate — the script is meant
to be copied to any Linux box, including ones where you cannot install anything,
so results from different machines are directly comparable.

> **It writes real data** to the directory you point it at, and competes with
> whatever else is running. Run it where that is acceptable. It refuses to start
> without enough free space and deletes its test file on the way out, including
> if you interrupt it.


## 1. Where the script is

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

# Published to /share on every Pipeline Airflow start, so it cannot drift from
# the copy in the repository.
SCRIPT = Path("/share/pipeline-airflow/lib/iobench.sh")
if not SCRIPT.exists():
    SCRIPT = Path("iobench.sh")   # or wherever you copied it

print("script:", SCRIPT, "-", "found" if SCRIPT.exists() else "NOT FOUND")
print("\nTo run it on another server, copy that one file across:")
print(f"  scp {SCRIPT} user@host:/tmp/ && ssh user@host 'sh /tmp/iobench.sh --dir /var/tmp'")


## 2. Run it here

`--dir` decides which device is measured — point it at the filesystem you care
about. Inside this add-on, `/data` and `/share` are on the Home Assistant data
disk, which is the one the pipeline writes to.

Start small. Raise `--size` if phases come out near zero seconds, which means the
work finished faster than the timer can resolve.


In [ ]:
RESULTS = Path("/share/pipeline-airflow/notebooks/io-results")
RESULTS.mkdir(parents=True, exist_ok=True)

def run_bench(label, target_dir="/data", size_mb=256, commits=200):
    """Run the script and return its parsed results, saving them under `label`."""
    out = RESULTS / f"{label}.json"
    proc = subprocess.run(
        ["sh", str(SCRIPT), "--dir", target_dir, "--size", str(size_mb),
         "--commits", str(commits), "--json", str(out)],
        capture_output=True, text=True,
    )
    print(proc.stdout or proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"iobench failed: {proc.stderr.strip()[:300]}")
    return json.loads(out.read_text())

here = run_bench("this-host", target_dir="/data", size_mb=256)


## 3. Reading the result

Two numbers carry the argument, and they are not the throughput:

- **busy%** — the share of wall-clock time the device had a request in flight.
  Near 100 means saturated: it had no idle capacity left to give you.
- **write wait** — average milliseconds per write. This is what actually rises
  when the device is the bottleneck, and what makes a pipeline feel slow.

The pairing is what distinguishes the two explanations:

| busy% | write wait | reading |
|---|---|---|
| high | high | **saturated** — the device is the limit |
| low | high | slow device, or one shared with something you cannot see |
| high | low | busy but keeping up — fine |
| low | low | the storage is not your problem; look at the workload |

Expect `commit` to show far lower MB/s than `bulk` on any device. That is not a
fault: fsynced 8 KiB writes are latency-bound and bulk writes are
throughput-bound, and a pipeline that commits often lives in the first world.


In [ ]:
def show(results, label=""):
    print(f"{label:<14} {'phase':<8} {'s':>7} {'MB/s':>9} {'busy%':>7} {'IOPS':>8} {'wr wait':>9}")
    for p in results["phases"]:
        print(f"{'':<14} {p['phase']:<8} {p['seconds']:>7.2f} {p['mb_s']:>9.1f} "
              f"{p['util_percent']:>7.1f} {p['iops']:>8.0f} {p['write_wait_ms']:>7.1f}ms")

show(here, "this host")

bulk = next(p for p in here["phases"] if p["phase"] == "bulk")
commit = next(p for p in here["phases"] if p["phase"] == "commit")
if commit["mb_s"]:
    print(f"\nbulk is {bulk['mb_s'] / commit['mb_s']:.0f}x the throughput of fsynced commits")
if bulk["util_percent"] == 0:
    print("busy% is 0 — /proc/diskstats was unavailable, so only the timings are real.")


## 4. Catching the slow window

The interesting run is the one taken **while it is slow**. Run the same command
during the working-hours window and compare it against a quiet baseline: same
workload, same size, so any difference is the device rather than the work.

Save each run under a label and compare them below.


In [ ]:
# Take one now, and another during the slow window:
#   quiet   = run_bench("quiet-0800", size_mb=256)
#   busy    = run_bench("busy-1030",  size_mb=256)

def compare(*labels):
    runs = {}
    for label in labels:
        path = RESULTS / f"{label}.json"
        if path.exists():
            runs[label] = json.loads(path.read_text())
        else:
            print(f"(no run saved as {label!r})")
    if len(runs) < 2:
        print("save at least two runs to compare")
        return
    phases = [p["phase"] for p in next(iter(runs.values()))["phases"]]
    print(f"{'phase':<8}" + "".join(f"{l:>22}" for l in runs))
    for phase in phases:
        row = f"{phase:<8}"
        for run in runs.values():
            p = next(x for x in run["phases"] if x["phase"] == phase)
            row += f"{p['mb_s']:>10.1f}MB/s{p['write_wait_ms']:>8.1f}ms"
        print(row)

compare("quiet-0800", "busy-1030")


## 5. Comparing servers

Copy `iobench.sh` to another machine, run it with the **same** `--size` and
`--commits`, and bring the JSON back here. Identical workload on different
storage is the cleanest comparison you can make.

```sh
scp iobench.sh user@other:/tmp/
ssh user@other 'sh /tmp/iobench.sh --dir /var/tmp --size 256 --json /tmp/other.json'
scp user@other:/tmp/other.json ./io-results/other-server.json
```

Two cautions when comparing. Check the `cache:` line in each run — a host without
O_DIRECT includes the page cache and will look far faster than it is. And a run
taken while the machine is busy measures contention, not the device, which is the
point when hunting the slow window and a distortion when comparing hardware.


In [ ]:
compare("this-host", "other-server")


---

### What this does and does not show

**Does**: how this storage behaves under a refresh-shaped load, right now,
including how much worse it gets when bulk writes and commits contend.

**Does not**: what your actual DAG does. This is a synthetic stand-in, sized by
you rather than by the data. It also cannot see *what else* is using the disk —
if utilisation is high while your pipeline is idle, something outside this
machine's view is responsible, and that is the finding rather than a gap.

For continuous capture rather than a point measurement, the **Add-on Watchdog**
samples the same counters every ten seconds and publishes them as Home Assistant
sensors, so the recorder keeps the history and a slow window is caught whether or
not anyone is watching.
